In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Preprocessing

In [ ]:
import pandas as pd
import json

project_dir = '/content/drive/MyDrive/VisualReview'

print("Loading reviews...")
reviews = []
with open(f'{project_dir}/Appliances.jsonl/Appliances.jsonl') as f:
    for line in f:
        reviews.append(json.loads(line))
reviews_df = pd.DataFrame(reviews)
print(f"Reviews loaded: {len(reviews_df):,}")

print("Loading metadata...")
meta = []
with open(f'{project_dir}/meta_Appliances.jsonl/meta_Appliances.jsonl') as f:
    for line in f:
        meta.append(json.loads(line))
meta_df = pd.DataFrame(meta)
print(f"Meta loaded: {len(meta_df):,}")

Loading reviews...
Reviews loaded: 2,128,605
Loading metadata...
Meta loaded: 94,327


In [ ]:
# Cell 2: Inspect both dataframes
print("=== REVIEWS COLUMNS ===")
print(reviews_df.columns.tolist())
print("\n=== REVIEWS SAMPLE ===")
print(reviews_df.head(2))

print("\n=== META COLUMNS ===")
print(meta_df.columns.tolist())
print("\n=== META SAMPLE ===")
print(meta_df.head(2))

=== REVIEWS COLUMNS ===
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

=== REVIEWS SAMPLE ===
   rating              title                                   text images  \
0     5.0         Work great  work great. use a new one every month     []   
1     5.0  excellent product                Little on the thin side     []   

         asin parent_asin                       user_id      timestamp  \
0  B01N0TQ0OH  B01N0TQ0OH  AGKHLEW2SOWHNMFQIJGBECAF7INQ  1519317108692   
1  B07DD2DMXB  B07DD37QPZ  AHWWLSPCJMALVHDDVSUGICL6RUCA  1664746863446   

   helpful_vote  verified_purchase  
0             0               True  
1             0               True  

=== META COLUMNS ===
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

=== META SAMPLE ===
          

In [ ]:
# Cell 3 (FIXED): Keep only columns we need
reviews_clean = reviews_df[[
    'parent_asin',
    'rating',
    'text',
]].copy()

meta_clean = meta_df[[
    'parent_asin',
    'title',
    'images',
    'main_category',
]].copy()

print("Reviews shape:", reviews_clean.shape)
print("Meta shape:", meta_clean.shape)

Reviews shape: (2128605, 3)
Meta shape: (94327, 4)


In [ ]:
# Cell 4 (FIXED): Extract image URL — images is a LIST of dicts
def extract_image_url(images):
    try:
        if not images or len(images) == 0:
            return None
        # each item is a dict with thumb, large, hi_res, variant
        # find the MAIN image first, fall back to first available
        for img in images:
            if img.get('variant') == 'MAIN':
                return img.get('hi_res') or img.get('large') or None
        # fallback: just return first image's hi_res or large
        first = images[0]
        return first.get('hi_res') or first.get('large') or None
    except:
        return None

meta_clean = meta_clean.copy()
meta_clean['image_url'] = meta_clean['images'].apply(extract_image_url)

print("Products with valid image URL:", meta_clean['image_url'].notna().sum())
print("Products without image URL:", meta_clean['image_url'].isna().sum())
print("\nSample image URL:", meta_clean['image_url'].dropna().iloc[0])

Products with valid image URL: 94326
Products without image URL: 1

Sample image URL: https://m.media-amazon.com/images/I/61zNIJh6ZCL._SL1500_.jpg


In [ ]:
# Cell 5 (FIXED): Join reviews with metadata
merged_df = reviews_clean.merge(
    meta_clean[['parent_asin', 'image_url', 'title']],
    on='parent_asin',
    how='inner'
)

print("Merged shape:", merged_df.shape)
print("\nSample row:")
print(merged_df.iloc[0])

Merged shape: (2128605, 5)

Sample row:
parent_asin                                           B01N0TQ0OH
rating                                                       5.0
text                       work great. use a new one every month
image_url      https://m.media-amazon.com/images/I/71zqD75W03...
title          Geesta 12-Pack Premium Activated Charcoal Wate...
Name: 0, dtype: object


In [ ]:
# Cell 6 (FIXED): Filter low quality records
print("Before filtering:", f"{len(merged_df):,}")

filtered_df = merged_df[
    merged_df['image_url'].notna() &
    merged_df['text'].notna() &
    (merged_df['text'].str.len() >= 50) &
    (merged_df['text'].str.len() <= 1000) &
    merged_df['rating'].notna()
].copy()

print("After filtering:", f"{len(filtered_df):,}")
print("\nRating distribution:")
print(filtered_df['rating'].value_counts().sort_index())

Before filtering: 2,128,605
After filtering: 1,391,466

Rating distribution:
rating
1.0    201596
2.0     66238
3.0     83407
4.0    144849
5.0    895376
Name: count, dtype: int64


In [ ]:
# Cell 7: Keep products with at least 3 reviews
reviews_per_product = filtered_df.groupby('parent_asin')['rating'].agg(
    count='count',
    unique_ratings=lambda x: x.nunique()
).reset_index()

good_products = reviews_per_product[reviews_per_product['count'] >= 3]['parent_asin']
filtered_df = filtered_df[filtered_df['parent_asin'].isin(good_products)].copy()

print(f"Products with 3+ reviews: {len(good_products):,}")
print(f"Total records remaining: {len(filtered_df):,}")

Products with 3+ reviews: 38,927
Total records remaining: 1,340,320


In [ ]:
# Cell 8: Save
import os
os.makedirs(f'{project_dir}/processed', exist_ok=True)

output_path = f'{project_dir}/processed/Appliances_joined.jsonl'
filtered_df.to_json(output_path, orient='records', lines=True)
print(f"Saved to {output_path}")
print(f"Final dataset: {len(filtered_df):,} review-image pairs")

Saved to /content/drive/MyDrive/VisualReview/processed/Appliances_joined.jsonl
Final dataset: 1,340,320 review-image pairs


# Image download through URLs

In [ ]:
# Cell 1: Check how many unique products we need to download images for
unique_products = filtered_df[['parent_asin', 'image_url']].drop_duplicates(subset='parent_asin')
print(f"Unique products to download images for: {len(unique_products):,}")
print("\nSample URLs:")
print(unique_products['image_url'].head(5).tolist())

Unique products to download images for: 38,927

Sample URLs:
['https://m.media-amazon.com/images/I/61fC-rtcH7L._AC_SL1500_.jpg', 'https://m.media-amazon.com/images/I/614VU14HAdL._AC_SL1200_.jpg', 'https://m.media-amazon.com/images/I/61S2sHE28UL._SL1500_.jpg', 'https://m.media-amazon.com/images/I/71meZeL4C5L._AC_SL1500_.jpg', 'https://m.media-amazon.com/images/I/61dbzuTcrRL._AC_SL1500_.jpg']


In [ ]:
# Cell 2: Create image directory
import os

image_dir = f'{project_dir}/data/images'
os.makedirs(image_dir, exist_ok=True)
print("Image directory created at:", image_dir)

Image directory created at: /content/drive/MyDrive/VisualReview/data/images


In [ ]:
# Cell 3 (FIXED): Download images with parallel processing + progress bar
import requests
import time
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def download_image(args):
    """Download a single image and save as {asin}.jpg"""
    asin, url, save_dir = args
    save_path = Path(save_dir) / f"{asin}.jpg"

    if save_path.exists():
        return asin, 'skipped'

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            return asin, 'success'
        else:
            return asin, f'failed_{response.status_code}'
    except Exception as e:
        return asin, f'error_{str(e)[:30]}'

# Prepare arguments
args_list = [
    (row['parent_asin'], row['image_url'], image_dir)
    for _, row in unique_products.iterrows()
]

results = {'success': 0, 'skipped': 0, 'failed': 0}
failed_asins = []

# 16 workers is a good balance — fast but won't get rate limited
NUM_WORKERS = 16

print(f"Downloading {len(args_list):,} images with {NUM_WORKERS} parallel workers...\n")

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(download_image, args): args for args in args_list}

    with tqdm(total=len(args_list), desc="Downloading images", unit="img") as pbar:
        for future in as_completed(futures):
            asin, status = future.result()

            if status == 'success':
                results['success'] += 1
            elif status == 'skipped':
                results['skipped'] += 1
            else:
                results['failed'] += 1
                failed_asins.append(asin)

            # Update progress bar with live stats
            pbar.set_postfix({
                'success': results['success'],
                'skipped': results['skipped'],
                'failed': results['failed']
            })
            pbar.update(1)

print("\n=== DOWNLOAD COMPLETE ===")
print(f"Successfully downloaded : {results['success']:,}")
print(f"Skipped (already existed): {results['skipped']:,}")
print(f"Failed                  : {results['failed']:,}")


=== DOWNLOAD COMPLETE ===
Successfully downloaded : 38,899
Skipped (already existed): 0
Failed                  : 28


In [ ]:
# Cell 4 (FIXED): Validate images in parallel + progress bar
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def validate_image(asin):
    path = f"{image_dir}/{asin}.jpg"
    try:
        img = Image.open(path)
        img.verify()
        return asin, 'valid'
    except Exception:
        return asin, 'corrupted'

asins_to_check = unique_products['parent_asin'].tolist()
corrupted = []
valid = 0

print(f"Validating {len(asins_to_check):,} images...\n")

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(validate_image, asin): asin for asin in asins_to_check}

    with tqdm(total=len(asins_to_check), desc="Validating images", unit="img") as pbar:
        for future in as_completed(futures):
            asin, status = future.result()
            if status == 'valid':
                valid += 1
            else:
                corrupted.append(asin)
            pbar.update(1)

print(f"\nValid images    : {valid:,}")
print(f"Corrupted/missing: {len(corrupted):,}")

Validating 38,927 images...



Validating images:   0%|          | 0/38927 [00:00<?, ?img/s]


Valid images    : 38,899
Corrupted/missing: 28


In [ ]:
# Cell 5: Remove reviews for products with failed/corrupted images
# so our final dataset only has rows where the image actually exists

good_asins = set(unique_products['parent_asin']) - set(corrupted) - set(failed_asins)

clean_df = filtered_df[filtered_df['parent_asin'].isin(good_asins)].copy()

# Add local image path column
clean_df['image_path'] = clean_df['parent_asin'].apply(
    lambda x: f"{image_dir}/{x}.jpg"
)

print(f"Final clean dataset: {len(clean_df):,} reviews")
print(f"Unique products: {clean_df['parent_asin'].nunique():,}")
print(f"\nSample row:")
print(clean_df.iloc[0])

Final clean dataset: 1,339,957 reviews
Unique products: 38,899

Sample row:
parent_asin                                           B078W2BJY8
rating                                                       5.0
text           I wasn't sure whether these were worth it or n...
image_url      https://m.media-amazon.com/images/I/61fC-rtcH7...
title          Filterlogic UKF8001 Water Filter, Replacement ...
image_path     /content/drive/MyDrive/VisualReview/data/image...
Name: 3, dtype: object


In [ ]:
# Cell 6: Save the final clean dataset
output_path = f'{project_dir}/processed/Appliances_final.jsonl'
clean_df.to_json(output_path, orient='records', lines=True)
print(f"Saved: {output_path}")
print(f"Total reviews: {len(clean_df):,}")
print(f"Total products with images: {clean_df['parent_asin'].nunique():,}")

Saved: /content/drive/MyDrive/VisualReview/processed/Appliances_final.jsonl
Total reviews: 1,339,957
Total products with images: 38,899


# Tokenizing the text using BPE

In [ ]:
# Cell 1: Load the final cleaned dataset
import pandas as pd
import json

project_dir = '/content/drive/MyDrive/VisualReview'

print("Loading final dataset...")
df = pd.read_json(f'{project_dir}/processed/Appliances_final.jsonl', lines=True)
print(f"Loaded: {len(df):,} reviews")
print(f"Rating distribution:\n{df['rating'].value_counts().sort_index()}")

Loading final dataset...
Loaded: 1,339,957 reviews
Rating distribution:
rating
1    191533
2     63526
3     79886
4    139713
5    865299
Name: count, dtype: int64


In [ ]:
# Cell 2 : Hard cap per rating class
per_class = 60_000  # 5 classes × 60K = 300K total
# (keeps 1-star which only has ~57K, caps the rest)

sampled_df = df.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), per_class), random_state=42),
    include_groups=True
).reset_index(drop=True)

print(f"Sampled dataset size: {len(sampled_df):,}")
print(f"\nRating distribution after sampling:")
print(sampled_df['rating'].value_counts().sort_index())
print(f"\nRating percentages:")
print((sampled_df['rating'].value_counts().sort_index() / len(sampled_df) * 100).round(2))

/tmp/ipykernel_61674/1841747212.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('rating', group_keys=False).apply(


Sampled dataset size: 300,000

Rating distribution after sampling:
rating
1    60000
2    60000
3    60000
4    60000
5    60000
Name: count, dtype: int64

Rating percentages:
rating
1    20.0
2    20.0
3    20.0
4    20.0
5    20.0
Name: count, dtype: float64


In [ ]:
# Cell 3: Save the sampled subset
output_path = f'{project_dir}/processed/Appliances_400k.jsonl'
sampled_df.to_json(output_path, orient='records', lines=True)
print(f"Saved: {output_path}")
print(f"Total reviews: {len(sampled_df):,}")
print(f"Unique products: {sampled_df['parent_asin'].nunique():,}")

Saved: /content/drive/MyDrive/VisualReview/processed/Appliances_400k.jsonl
Total reviews: 300,000
Unique products: 31,824


In [ ]:
# Cell 5: Train BPE tokenizer from scratch on review text
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

# Step 1: Initialize a blank BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Step 2: Define special tokens
# [UNK]  — unknown token for words not in vocabulary
# [PAD]  — padding token to make batches equal length
# [BOS]  — beginning of sequence
# [EOS]  — end of sequence
# [S1]-[S5] — sentiment control tokens (star ratings)
# [P1]-[P3] — persona control tokens (budget, casual, enthusiast)
# [A1]-[A5] — aspect control tokens (performance, design, value, durability, ease of use)

special_tokens = [
    "[UNK]", "[PAD]", "[BOS]", "[EOS]",
    "[S1]", "[S2]", "[S3]", "[S4]", "[S5]",   # sentiment
    "[P1]", "[P2]", "[P3]",                     # persona
    "[A1]", "[A2]", "[A3]", "[A4]", "[A5]",    # aspect
]

# Step 3: Configure the trainer
trainer = BpeTrainer(
    vocab_size=16000,          # vocabulary size — enough for review text
    min_frequency=3,           # ignore tokens appearing fewer than 3 times
    special_tokens=special_tokens,
    show_progress=True
)

# Step 4: Feed all review text to the trainer
# We write reviews to a temp txt file — that's what the trainer expects
import tempfile
import os

print("Writing reviews to temp file for tokenizer training...")
tmp_path = '/tmp/reviews_corpus.txt'
with open(tmp_path, 'w', encoding='utf-8') as f:
    for text in sampled_df['text'].dropna():
        f.write(text.strip() + '\n')

print(f"Corpus file written. Training BPE tokenizer on {len(sampled_df):,} reviews...")

# Step 5: Train
tokenizer.train(files=[tmp_path], trainer=trainer)
print("Tokenizer training complete!")
print(f"Vocabulary size: {tokenizer.get_vocab_size():,}")

Writing reviews to temp file for tokenizer training...
Corpus file written. Training BPE tokenizer on 300,000 reviews...
Tokenizer training complete!
Vocabulary size: 16,000


In [ ]:
# Cell 6: Add BOS/EOS post-processing so every encoded sequence
# automatically gets [BOS] prepended and [EOS] appended
tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]")),
    ],
)

# Quick sanity check
sample_text = "This product works great, very easy to install and use!"
encoded = tokenizer.encode(sample_text)
print("=== TOKENIZER SANITY CHECK ===")
print(f"Input text  : {sample_text}")
print(f"Tokens      : {encoded.tokens}")
print(f"Token IDs   : {encoded.ids}")
print(f"Decoded back: {tokenizer.decode(encoded.ids)}")

=== TOKENIZER SANITY CHECK ===
Input text  : This product works great, very easy to install and use!
Tokens      : ['[BOS]', 'This', 'product', 'works', 'great', ',', 'very', 'easy', 'to', 'install', 'and', 'use', '!', '[EOS]']
Token IDs   : [2, 600, 615, 702, 638, 30, 572, 705, 463, 614, 473, 556, 19, 3]
Decoded back: This product works great , very easy to install and use !


In [ ]:
# Cell 7: Save the tokenizer to Drive
tokenizer_path = f'{project_dir}/tokenizer/appliances_bpe_tokenizer.json'
os.makedirs(f'{project_dir}/tokenizer', exist_ok=True)
tokenizer.save(tokenizer_path)
print(f"Tokenizer saved to: {tokenizer_path}")

# Verify it reloads correctly
from tokenizers import Tokenizer
reloaded = Tokenizer.from_file(tokenizer_path)
print(f"Reloaded vocabulary size: {reloaded.get_vocab_size():,}")
print("Tokenizer ready!")

Tokenizer saved to: /content/drive/MyDrive/VisualReview/tokenizer/appliances_bpe_tokenizer.json
Reloaded vocabulary size: 16,000
Tokenizer ready!


In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<.*?>', ' ', text)      # remove HTML tags like <br />
    text = re.sub(r'\s+', ' ', text)         # collapse multiple spaces
    return text.strip()

train_df['text'] = train_df['text'].apply(clean_text)
val_df['text']   = val_df['text'].apply(clean_text)
test_df['text']  = test_df['text'].apply(clean_text)

print("HTML cleaning done!")

HTML cleaning done!


# Dataset classes

In [ ]:
# Cell 1: Mount Drive + set all path variables + global config
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tokenizers import Tokenizer
import pandas as pd
import numpy as np
import re

# ── Project paths ──────────────────────────────────────────────
project_dir    = '/content/drive/MyDrive/VisualReview'
image_dir      = f'{project_dir}/data/images'
tokenizer_path = f'{project_dir}/tokenizer/appliances_bpe_tokenizer.json'

processed_dir  = f'{project_dir}/processed'
train_path     = f'{processed_dir}/train.jsonl'
val_path       = f'{processed_dir}/val.jsonl'
test_path      = f'{processed_dir}/test.jsonl'
labeled_path   = f'{processed_dir}/Appliances_labeled.jsonl'

# ── Global model config ────────────────────────────────────────
BATCH_SIZE    = 64
EMBED_DIM     = 512
NUM_HEADS     = 8
NUM_LAYERS    = 6
FF_DIM        = 2048
MAX_LENGTH    = 256
IMAGE_SIZE    = 224
LEARNING_RATE = 3e-4
USE_AMP       = True
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Verify paths ───────────────────────────────────────────────
paths_to_check = {
    'Image dir'  : image_dir,
    'Tokenizer'  : tokenizer_path,
    'Train split': train_path,
    'Val split'  : val_path,
    'Test split' : test_path,
    'Labeled data': labeled_path,
}

print("=== PATH CHECK ===")
all_good = True
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ MISSING"
    print(f"{status}  {name}: {path}")
    if not exists:
        all_good = False

print()
if all_good:
    print("All paths verified — ready to go!")
else:
    print("Some paths are missing — check your Drive folder structure.")

# ── GPU check ──────────────────────────────────────────────────
print(f"\nDevice: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
=== PATH CHECK ===
✅  Image dir: /content/drive/MyDrive/VisualReview/data/images
✅  Tokenizer: /content/drive/MyDrive/VisualReview/tokenizer/appliances_bpe_tokenizer.json
❌ MISSING  Train split: /content/drive/MyDrive/VisualReview/processed/train.jsonl
❌ MISSING  Val split: /content/drive/MyDrive/VisualReview/processed/val.jsonl
❌ MISSING  Test split: /content/drive/MyDrive/VisualReview/processed/test.jsonl
❌ MISSING  Labeled data: /content/drive/MyDrive/VisualReview/processed/Appliances_labeled.jsonl

Some paths are missing — check your Drive folder structure.

Device: cuda
GPU    : NVIDIA A100-SXM4-40GB
Memory : 42.4 GB


In [ ]:
# Cell 2: Define persona assignment function
# Automatically assigns persona label to each review based on writing style
import pandas as pd
import os

# Ensure sampled_df is loaded (in case kernel restarted)
if 'sampled_df' not in locals():
    path_400k = '/content/drive/MyDrive/VisualReview/processed/Appliances_400k.jsonl'
    if os.path.exists(path_400k):
        print("Loading sampled_df from Drive...")
        sampled_df = pd.read_json(path_400k, lines=True)
    else:
        raise NameError("sampled_df not found. Please run the sampling cell (Cell 2) or ensure the file exists on Drive.")

BUDGET_KEYWORDS = [
    'price', 'cheap', 'expensive', 'cost', 'worth', 'value',
    'affordable', 'overpriced', 'deal', 'money', 'budget', 'paid'
]

ENTHUSIAST_KEYWORDS = [
    'performance', 'specs', 'watt', 'voltage', 'capacity', 'efficiency',
    'motor', 'rpm', 'power', 'technical', 'rating', 'btu', 'decibel',
    'compressor', 'filter', 'sensor', 'cycle', 'ampere', 'temperature'
]

def assign_persona(text):
    if not isinstance(text, str):
        return 'P3'

    text_lower = text.lower()
    budget_score = sum(1 for kw in BUDGET_KEYWORDS if kw in text_lower)
    enthusiast_score = sum(1 for kw in ENTHUSIAST_KEYWORDS if kw in text_lower)

    if budget_score == 0 and enthusiast_score == 0:
        return 'P3'
    elif budget_score >= enthusiast_score:
        return 'P1'
    else:
        return 'P2'

print("Assigning persona labels...")
sampled_df['persona'] = sampled_df['text'].apply(assign_persona)

print("Persona distribution:")
print(sampled_df['persona'].value_counts())
print("\nPersona percentages:")
print((sampled_df['persona'].value_counts() / len(sampled_df) * 100).round(2))

Loading sampled_df from Drive...
Assigning persona labels...
Persona distribution:
persona
P3    179980
P1     68735
P2     51285
Name: count, dtype: int64

Persona percentages:
persona
P3    59.99
P1    22.91
P2    17.10
Name: count, dtype: float64


In [ ]:
# Cell 3: Define aspect assignment function
# Assigns one of 5 aspect labels based on what the review mainly talks about

ASPECT_KEYWORDS = {
    'A1': ['performance', 'power', 'speed', 'efficient', 'effective',
           'works', 'function', 'motor', 'strong', 'capacity'],         # Performance
    'A2': ['design', 'look', 'color', 'style', 'appearance', 'sleek',
           'beautiful', 'ugly', 'aesthetic', 'finish', 'compact'],      # Design
    'A3': ['price', 'value', 'worth', 'cost', 'cheap', 'expensive',
           'affordable', 'money', 'deal', 'budget'],                     # Value
    'A4': ['durable', 'quality', 'build', 'sturdy', 'solid', 'broke',
           'lasted', 'cheap', 'flimsy', 'material', 'construction'],    # Durability
    'A5': ['easy', 'simple', 'install', 'setup', 'instructions',
           'difficult', 'complicated', 'user', 'intuitive', 'manual'],  # Ease of use
}

def assign_aspect(text):
    """
    Assigns the dominant aspect label based on keyword frequency.
    Falls back to A1 (Performance) if no keywords match.
    """
    if not isinstance(text, str):
        return 'A1'

    text_lower = text.lower()
    scores = {
        aspect: sum(1 for kw in keywords if kw in text_lower)
        for aspect, keywords in ASPECT_KEYWORDS.items()
    }

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'A1'

print("Assigning aspect labels...")
sampled_df['aspect'] = sampled_df['text'].apply(assign_aspect)

print("Aspect distribution:")
print(sampled_df['aspect'].value_counts())
print("\nAspect percentages:")
print((sampled_df['aspect'].value_counts() / len(sampled_df) * 100).round(2))

Assigning aspect labels...
Aspect distribution:
aspect
A1    159546
A3     47387
A5     38572
A2     27503
A4     26992
Name: count, dtype: int64

Aspect percentages:
aspect
A1    53.18
A3    15.80
A5    12.86
A2     9.17
A4     9.00
Name: count, dtype: float64


In [ ]:
# Cell 4: Save the dataframe with persona and aspect labels
output_path = f'{project_dir}/processed/Appliances_labeled.jsonl'
sampled_df.to_json(output_path, orient='records', lines=True)
print(f"Saved labeled dataset to: {output_path}")

Saved labeled dataset to: /content/drive/MyDrive/VisualReview/processed/Appliances_labeled.jsonl


In [ ]:
# Cell 5: Train / Val / Test split
from sklearn.model_selection import train_test_split

# Split at product level — not review level
# This ensures the same product doesn't appear in both train and val
# which would be data leakage
unique_asins = sampled_df['parent_asin'].unique()

train_asins, temp_asins = train_test_split(unique_asins, test_size=0.2, random_state=42)
val_asins, test_asins = train_test_split(temp_asins, test_size=0.5, random_state=42)

train_df = sampled_df[sampled_df['parent_asin'].isin(train_asins)].reset_index(drop=True)
val_df   = sampled_df[sampled_df['parent_asin'].isin(val_asins)].reset_index(drop=True)
test_df  = sampled_df[sampled_df['parent_asin'].isin(test_asins)].reset_index(drop=True)

print(f"Train reviews : {len(train_df):,} | Products: {train_df['parent_asin'].nunique():,}")
print(f"Val reviews   : {len(val_df):,} | Products: {val_df['parent_asin'].nunique():,}")
print(f"Test reviews  : {len(test_df):,} | Products: {test_df['parent_asin'].nunique():,}")

# Save splits
train_df.to_json(f'{project_dir}/processed/train.jsonl', orient='records', lines=True)
val_df.to_json(f'{project_dir}/processed/val.jsonl',   orient='records', lines=True)
test_df.to_json(f'{project_dir}/processed/test.jsonl', orient='records', lines=True)
print("\nAll splits saved!")

Train reviews : 240,744 | Products: 25,459
Val reviews   : 29,717 | Products: 3,182
Test reviews  : 29,539 | Products: 3,183

All splits saved!


In [ ]:
# Cell 6: Define the PyTorch Dataset class
class ProductReviewDataset(Dataset):
    def __init__(self, df, tokenizer_path, image_dir, max_length=256):
        """
        Args:
            df            : dataframe with columns — parent_asin, rating,
                            text, persona, aspect, image_path
            tokenizer_path: path to saved BPE tokenizer json
            image_dir     : folder where {asin}.jpg files live
            max_length    : max token length for reviews (256)
        """
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = Tokenizer.from_file(tokenizer_path)
        self.image_dir  = image_dir
        self.max_length = max_length

        # Map control tokens to their IDs in the vocabulary
        self.sentiment_map = {
            1.0: '[S1]', 2.0: '[S2]', 3.0: '[S3]',
            4.0: '[S4]', 5.0: '[S5]'
        }
        self.persona_map = {
            'P1': '[P1]', 'P2': '[P2]', 'P3': '[P3]'
        }
        self.aspect_map = {
            'A1': '[A1]', 'A2': '[A2]', 'A3': '[A3]',
            'A4': '[A4]', 'A5': '[A5]'
        }

        # Standard ImageNet normalization for ResNet
        self.image_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        self.pad_id = self.tokenizer.token_to_id('[PAD]')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- 1. Load and transform image ---
        image_path = os.path.join(self.image_dir, f"{row['parent_asin']}.jpg")
        try:
            image = Image.open(image_path).convert('RGB')
            image = self.image_transform(image)
        except Exception:
            # If image is missing/corrupted return a blank tensor
            image = torch.zeros(3, 224, 224)

        # --- 2. Build control token prefix ---
        # Format: [S?] [P?] [A?] — prepended before the review text
        sentiment_token = self.sentiment_map.get(row['rating'], '[S3]')
        persona_token   = self.persona_map.get(row['persona'], '[P3]')
        aspect_token    = self.aspect_map.get(row['aspect'], '[A1]')

        control_prefix = f"{sentiment_token} {persona_token} {aspect_token}"

        # --- 3. Tokenize review text ---
        full_text = control_prefix + ' ' + str(row['text'])
        encoded   = self.tokenizer.encode(full_text)
        token_ids = encoded.ids

        # --- 4. Truncate or pad to max_length ---
        if len(token_ids) > self.max_length:
            token_ids = token_ids[:self.max_length]
        else:
            token_ids = token_ids + [self.pad_id] * (self.max_length - len(token_ids))

        token_ids = torch.tensor(token_ids, dtype=torch.long)

        # --- 5. Build attention mask (1 for real tokens, 0 for padding) ---
        attention_mask = (token_ids != self.pad_id).long()

        # --- 6. Build labels for language modeling ---
        # Labels are the same as token_ids but with PAD positions set to -100
        # -100 tells PyTorch's CrossEntropyLoss to ignore those positions
        labels = token_ids.clone()
        labels[labels == self.pad_id] = -100

        return {
            'image'         : image,           # (3, 224, 224)
            'input_ids'     : token_ids,        # (256,)
            'attention_mask': attention_mask,   # (256,)
            'labels'        : labels,           # (256,)
        }

In [ ]:
# Cell 7: Test the Dataset class end to end
tokenizer_path = f'{project_dir}/tokenizer/appliances_bpe_tokenizer.json'
image_dir      = f'{project_dir}/data/images'

# Test on a small slice of train_df first
test_dataset = ProductReviewDataset(
    df             = train_df.head(100),
    tokenizer_path = tokenizer_path,
    image_dir      = image_dir,
    max_length     = 256
)

print(f"Dataset length: {len(test_dataset)}")

# Grab one sample and check shapes
sample = test_dataset[0]
print(f"\nimage shape          : {sample['image'].shape}")
print(f"input_ids shape      : {sample['input_ids'].shape}")
print(f"attention_mask shape : {sample['attention_mask'].shape}")
print(f"labels shape         : {sample['labels'].shape}")

# Decode the tokens back to text so we can visually verify
tokenizer = Tokenizer.from_file(tokenizer_path)
decoded = tokenizer.decode(sample['input_ids'].tolist())
print(f"\nDecoded input:\n{decoded[:300]}")

Dataset length: 100

image shape          : torch.Size([3, 224, 224])
input_ids shape      : torch.Size([256])
attention_mask shape : torch.Size([256])
labels shape         : torch.Size([256])

Decoded input:
Replacement rollers too narrow for track on my < br /> LG dishwasher they look correct but aren ' t < br /> Bought from different vender paid more fit correctly .


In [ ]:
# Cell 8: Test the DataLoader with batching — updated for A100
from torch.cuda.amp import GradScaler, autocast

train_loader = DataLoader(
    test_dataset,
    batch_size=64,        # up from 4
    shuffle=True,
    num_workers=4,        # up from 2
    pin_memory=True,
    prefetch_factor=2
)

batch = next(iter(train_loader))
print("=== BATCH SHAPES ===")
print(f"image          : {batch['image'].shape}")           # (64, 3, 224, 224)
print(f"input_ids      : {batch['input_ids'].shape}")       # (64, 256)
print(f"attention_mask : {batch['attention_mask'].shape}")  # (64, 256)
print(f"labels         : {batch['labels'].shape}")          # (64, 256)

# Confirm tensors move to GPU correctly
batch_image = batch['image'].to(DEVICE)
print(f"\nImage tensor on : {batch_image.device}")
print(f"Mixed precision : {'enabled (fp16)' if USE_AMP else 'disabled'}")
print("\nDataLoader working correctly!")

=== BATCH SHAPES ===
image          : torch.Size([64, 3, 224, 224])
input_ids      : torch.Size([64, 256])
attention_mask : torch.Size([64, 256])
labels         : torch.Size([64, 256])

Image tensor on : cuda:0
Mixed precision : enabled (fp16)

DataLoader working correctly!


# Model training

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Restating lost global variables ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMBED_DIM = 512

class ResidualBlock(nn.Module):
    """
    Basic residual block: two 3×3 convs with a skip connection.
    If in_channels != out_channels, the skip is projected with a 1×1 conv.
    BatchNorm + ReLU after each conv, ReLU after the addition.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        # Skip projection when dimensions change
        self.skip = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if (stride != 1 or in_channels != out_channels) else nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.skip(x))


class ImageEncoder(nn.Module):
    """
    ResNet-style CNN built from scratch.

    Input : (B, 3, 224, 224)
    Output: (B, 49, 512)  — 7×7 spatial grid, each cell is a 512-dim vector

    Architecture:
      conv stem  : 7×7, stride 2 → (B, 64, 112, 112)
      maxpool    : stride 2       → (B, 64,  56,  56)
      layer 1    : 2× ResBlock 64  → 64,  56×56
      layer 2    : 2× ResBlock 128 → 128, 28×28  (stride-2 in first block)
      layer 3    : 2× ResBlock 256 → 256, 14×14  (stride-2 in first block)
      layer 4    : 2× ResBlock 512 → 512,  7×7   (stride-2 in first block)
      flatten    : (B, 512, 7, 7) → (B, 49, 512)
    """
    def __init__(self, embed_dim=512):
        super().__init__()

        # ── Stem ──────────────────────────────────────────
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),  # → 56×56
        )

        # ── Residual stages ───────────────────────────────
        self.layer1 = self._make_layer(64,  64,  n_blocks=2, stride=1)  # 56×56
        self.layer2 = self._make_layer(64,  128, n_blocks=2, stride=2)  # 28×28
        self.layer3 = self._make_layer(128, 256, n_blocks=2, stride=2)  # 14×14
        self.layer4 = self._make_layer(256, 512, n_blocks=2, stride=2)  #  7×7

        # ── Final projection to EMBED_DIM (no-op here, both are 512) ──
        # Kept as a named layer so it's easy to change EMBED_DIM later
        self.proj = nn.Conv2d(512, embed_dim, kernel_size=1, bias=False)
        self.proj_norm = nn.BatchNorm2d(embed_dim)

        self._init_weights()

    def _make_layer(self, in_ch, out_ch, n_blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride=stride)]
        for _ in range(1, n_blocks):
            layers.append(ResidualBlock(out_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                        nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: (B, 3, 224, 224)
        x = self.stem(x)          # → (B,  64, 56, 56)
        x = self.layer1(x)        # → (B,  64, 56, 56)
        x = self.layer2(x)        # → (B, 128, 28, 28)
        x = self.layer3(x)        # → (B, 256, 14, 14)
        x = self.layer4(x)        # → (B, 512,  7,  7)
        x = F.relu(self.proj_norm(self.proj(x)))  # → (B, 512, 7, 7)

        B, C, H, W = x.shape      # H=W=7, C=512
        x = x.flatten(2)          # → (B, 512, 49)
        x = x.permute(0, 2, 1)    # → (B, 49, 512)
        return x


# ── Quick shape sanity check ──────────────────────────────
encoder = ImageEncoder(embed_dim=EMBED_DIM).to(DEVICE)
dummy_img = torch.randn(2, 3, 224, 224).to(DEVICE)
out = encoder(dummy_img)
print(f"Encoder output: {out.shape}")  # expect (2, 49, 512)

total_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder parameters: {total_params:,}")

Encoder output: torch.Size([2, 49, 512])
Encoder parameters: 11,439,680


In [2]:
class CrossAttentionBridge(nn.Module):
    """
    Projects image encoder output into decoder embedding space and
    normalises it so the decoder's cross-attention has stable keys/values.

    Input : (B, 49, 512)  — raw spatial features from the CNN
    Output: (B, 49, 512)  — projected + normalised image tokens
                            (these become K and V in every decoder layer)
    """
    def __init__(self, image_dim=512, embed_dim=512):
        super().__init__()
        # Linear projection (no-op when dims match, but makes the
        # architecture explicit and easy to change)
        self.proj = nn.Linear(image_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, image_features):
        # image_features: (B, 49, 512)
        projected = self.proj(image_features)   # (B, 49, embed_dim)
        return self.norm(projected)             # (B, 49, embed_dim)


# ── Quick check ───────────────────────────────────────────
bridge = CrossAttentionBridge(image_dim=EMBED_DIM,
                              embed_dim=EMBED_DIM).to(DEVICE)
bridge_out = bridge(out)   # out is still (2, 49, 512) from encoder check
print(f"Bridge output: {bridge_out.shape}")  # expect (2, 49, 512)

Bridge output: torch.Size([2, 49, 512])


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Restating lost global variables ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMBED_DIM = 512
NUM_HEADS = 8
NUM_LAYERS = 6
FF_DIM = 2048
MAX_LENGTH = 256

class MultiHeadSelfAttention(nn.Module):
    """Causal (masked) multi-head self-attention."""
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5

        self.qkv  = nn.Linear(embed_dim, 3 * embed_dim)
        self.out  = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, causal_mask):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.masked_fill(causal_mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)

        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5

        self.q_proj   = nn.Linear(embed_dim, embed_dim)
        self.kv_proj  = nn.Linear(embed_dim, 2 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, image_kv):
        B, T, C = x.shape
        _, S, _ = image_kv.shape

        q  = self.q_proj(x).reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        kv = self.kv_proj(image_kv).reshape(B, S, 2, self.num_heads, self.head_dim)
        kv = kv.permute(2, 0, 3, 1, 4)
        k, v = kv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)

        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

class FeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim),
        )
    def forward(self, x): return self.net(x)

class DecoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.self_attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.cross_attn = MultiHeadCrossAttention(embed_dim, num_heads)
        self.ln3 = nn.LayerNorm(embed_dim)
        self.ff = FeedForward(embed_dim, ff_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, image_kv, causal_mask):
        x = x + self.drop(self.self_attn(self.ln1(x), causal_mask))
        x = x + self.drop(self.cross_attn(self.ln2(x), image_kv))
        x = x + self.drop(self.ff(self.ln3(x)))
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, ff_dim, max_length, dropout=0.1):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_length, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([DecoderLayer(embed_dim, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.final_ln = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight

    def _causal_mask(self, seq_len, device):
        return torch.tril(torch.ones(1, 1, seq_len, seq_len, device=device)).bool()

    def forward(self, input_ids, image_kv):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.drop(self.token_embed(input_ids) + self.pos_embed(positions))
        causal_mask = self._causal_mask(T, input_ids.device)
        for layer in self.layers: x = layer(x, image_kv, causal_mask)
        return self.lm_head(self.final_ln(x))

# Quick Check
decoder = Decoder(16000, EMBED_DIM, NUM_HEADS, NUM_LAYERS, FF_DIM, MAX_LENGTH).to(DEVICE)
dummy_ids = torch.randint(0, 16000, (2, 256)).to(DEVICE)
dummy_kv = torch.randn(2, 49, EMBED_DIM).to(DEVICE)
logits = decoder(dummy_ids, dummy_kv)
print(f"Decoder logits: {logits.shape}")

Decoder logits: torch.Size([2, 256, 16000])


In [5]:
class VisualReviewModel(nn.Module):
    """
    Full model: ImageEncoder → CrossAttentionBridge → Decoder.
    Single forward pass takes an image + input token sequence,
    returns next-token logits for the full sequence.
    """
    def __init__(self, vocab_size, embed_dim, num_heads,
                 num_layers, ff_dim, max_length, dropout=0.1):
        super().__init__()
        self.encoder = ImageEncoder(embed_dim=embed_dim)
        self.bridge  = CrossAttentionBridge(image_dim=embed_dim,
                                            embed_dim=embed_dim)
        self.decoder = Decoder(vocab_size, embed_dim, num_heads,
                               num_layers, ff_dim, max_length, dropout)

    def forward(self, images, input_ids):
        # images   : (B, 3, 224, 224)
        # input_ids: (B, T)

        image_features = self.encoder(images)     # (B, 49, embed_dim)
        image_kv       = self.bridge(image_features)  # (B, 49, embed_dim)
        logits         = self.decoder(input_ids, image_kv)  # (B, T, vocab_size)
        return logits


# ── Full model sanity check ───────────────────────────────
model = VisualReviewModel(
    vocab_size  = 16000,
    embed_dim   = EMBED_DIM,
    num_heads   = NUM_HEADS,
    num_layers  = NUM_LAYERS,
    ff_dim      = FF_DIM,
    max_length  = MAX_LENGTH,
    dropout     = 0.1,
).to(DEVICE)

dummy_img  = torch.randn(2, 3, 224, 224).to(DEVICE)
dummy_ids  = torch.randint(0, 16000, (2, 256)).to(DEVICE)

with torch.no_grad():
    logits = model(dummy_img, dummy_ids)

print(f"Full model output: {logits.shape}")  # expect (2, 256, 16000)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

Full model output: torch.Size([2, 256, 16000])
Total parameters    : 45,251,648
Trainable parameters: 45,251,648
